# DDL Gold - Definiciones de Estructura

**Propósito**: Define la infraestructura de la capa Gold (esquema, dimensiones, fact, vista analítica).

**Ejecutar**: Solo la primera vez o cuando cambie la estructura del modelo dimensional.

**Modelo**: Esquema Estrella (Star Schema)
* 4 Dimensiones: dim_marca, dim_tipo_vehiculo, dim_modelo, dim_geografia
* 1 Fact: fact_transferencias
* 1 Vista analítica: vw_transferencias_analitica (JOIN de fact + dimensiones)

**Notas**:
* El notebook ETL `04_Gold` se encarga de CARGAR los datos (INSERT OVERWRITE)

In [0]:
%sql
-- Crear el esquema Gold si no existe
CREATE SCHEMA IF NOT EXISTS workspace.tp_dnrpa_gold
COMMENT 'Capa Gold - Modelo Dimensional (Esquema Estrella) para análisis de transferencias vehiculares';

In [0]:
%sql
-- Crear tabla de dimensión: dim_marca
CREATE TABLE IF NOT EXISTS workspace.tp_dnrpa_gold.dim_marca (
  automotor_marca_codigo STRING COMMENT 'Código de marca (PK)',
  automotor_marca_descripcion STRING COMMENT 'Nombre de la marca'
) USING DELTA
COMMENT 'Dimensión de marcas de vehículos';

In [0]:
%sql
-- Crear tabla de dimensión: dim_tipo_vehiculo
CREATE TABLE IF NOT EXISTS workspace.tp_dnrpa_gold.dim_tipo_vehiculo (
  automotor_tipo_codigo STRING COMMENT 'Código de tipo (PK)',
  automotor_tipo_descripcion STRING COMMENT 'Descripción del tipo'
) USING DELTA
COMMENT 'Dimensión de tipos de vehículos';

In [0]:
%sql
-- Crear tabla de dimensión: dim_modelo
CREATE TABLE IF NOT EXISTS workspace.tp_dnrpa_gold.dim_modelo (
  automotor_modelo_codigo STRING COMMENT 'Código de modelo (PK)',
  automotor_modelo_descripcion STRING COMMENT 'Nombre del modelo'
) USING DELTA
COMMENT 'Dimensión de modelos de vehículos';

In [0]:
%sql
-- Crear tabla de dimensión: dim_geografia
CREATE TABLE IF NOT EXISTS workspace.tp_dnrpa_gold.dim_geografia (
  registro_seccional_codigo STRING COMMENT 'Código del registro seccional (PK)',
  registro_seccional_provincia STRING COMMENT 'Provincia del registro'
) USING DELTA
COMMENT 'Dimensión geográfica (seccionales y provincias)';

In [0]:
%sql
-- Crear tabla de hechos: fact_transferencias
CREATE TABLE IF NOT EXISTS workspace.tp_dnrpa_gold.fact_transferencias (
  id_tramite STRING COMMENT 'Clave sintética única (PK)',
  tramite_fecha DATE COMMENT 'Fecha del trámite',
  automotor_anio_modelo STRING COMMENT 'Año modelo del vehículo (métrica)',
  automotor_marca_codigo STRING COMMENT 'Código de marca (FK a dim_marca)',
  automotor_tipo_codigo STRING COMMENT 'Código de tipo (FK a dim_tipo_vehiculo)',
  automotor_modelo_codigo STRING COMMENT 'Código de modelo (FK a dim_modelo)',
  registro_seccional_codigo STRING COMMENT 'Código del registro (FK a dim_geografia)'
) USING DELTA
COMMENT 'Tabla de hechos de transferencias vehiculares';

In [0]:
%sql
-- Crear vista analítica desnormalizada
-- JOIN automático de fact con las 4 dimensiones
CREATE OR REPLACE VIEW workspace.tp_dnrpa_gold.vw_transferencias_analitica
COMMENT 'Vista analítica desnormalizada: fact + dimensiones (listo para BI)'
AS
SELECT
  -- Dimensión temporal
  f.tramite_fecha,
  YEAR(f.tramite_fecha) AS anio_tramite,
  MONTH(f.tramite_fecha) AS mes_tramite,
  
  -- Dimensión geográfica
  g.registro_seccional_codigo,
  g.registro_seccional_provincia,
  
  -- Dimensión marca
  m.automotor_marca_codigo,
  m.automotor_marca_descripcion,
  
  -- Dimensión tipo
  t.automotor_tipo_codigo,
  t.automotor_tipo_descripcion,
  
  -- Dimensión modelo
  mo.automotor_modelo_codigo,
  mo.automotor_modelo_descripcion,
  
  -- Métricas
  f.automotor_anio_modelo,
  f.id_tramite
  
FROM workspace.tp_dnrpa_gold.fact_transferencias f
LEFT JOIN workspace.tp_dnrpa_gold.dim_geografia g ON f.registro_seccional_codigo = g.registro_seccional_codigo
LEFT JOIN workspace.tp_dnrpa_gold.dim_marca m ON f.automotor_marca_codigo = m.automotor_marca_codigo
LEFT JOIN workspace.tp_dnrpa_gold.dim_tipo_vehiculo t ON f.automotor_tipo_codigo = t.automotor_tipo_codigo
LEFT JOIN workspace.tp_dnrpa_gold.dim_modelo mo ON f.automotor_modelo_codigo = mo.automotor_modelo_codigo;

## ✅ Verificación de Infraestructura

Después de ejecutar todas las celdas anteriores, verifica que la infraestructura se creó correctamente:

```sql
-- Ver todas las tablas y vistas creadas
SHOW TABLES IN workspace.tp_dnrpa_gold;
```

**Resultado esperado**: 5 tablas + 1 vista
* dim_geografia
* dim_marca
* dim_modelo
* dim_tipo_vehiculo
* fact_transferencias
* vw_transferencias_analitica

---

## 📖 Uso de la Vista Analítica

La vista `vw_transferencias_analitica` está lista para consultas de BI sin necesidad de escribir JOINs:

### Ejemplo 1: Top marcas por provincia
```sql
SELECT 
  registro_seccional_provincia,
  automotor_marca_descripcion,
  COUNT(*) as transferencias
FROM workspace.tp_dnrpa_gold.vw_transferencias_analitica
GROUP BY 1, 2
ORDER BY 1, 3 DESC;
```

### Ejemplo 2: Evolución mensual
```sql
SELECT 
  anio_tramite,
  mes_tramite,
  COUNT(*) as transferencias
FROM workspace.tp_dnrpa_gold.vw_transferencias_analitica
GROUP BY 1, 2
ORDER BY 1 DESC, 2 DESC;
```

---

## 🔄 Siguiente Paso

Ejecutar los notebooks ETL para cargar datos:
1. `02_Bronze_dnrpa` → Carga datos crudos
2. `02_Silver_dnrpa` → Transforma y limpia
3. `04_Gold` → Carga dimensiones y fact

In [0]:
%sql
-- Verificar que todas las tablas y vistas se crearon correctamente
SHOW TABLES IN workspace.tp_dnrpa_gold;